In [ ]:
!pip install -q polars faiss-cpu

In [ ]:
import os
import gc
import shutil
import pandas as pd
import numpy as np
import polars as pl
import torch
import torch.nn as nn
import torch.nn.functional as F
import scipy.sparse as sp
import faiss
from tqdm.auto import tqdm

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Đang sử dụng thiết bị: {device}")

In [ ]:
DATASET_DIR_NAME = 'datasets/b22dckh072/train-t' 

INPUT_DIR = f'/kaggle/input/{DATASET_DIR_NAME}'
WORKING_DIR = '/kaggle/working'

# Khai báo đường dẫn file
TRAIN_PATH = os.path.join(INPUT_DIR, 'train_interactions.parquet')

# Đường dẫn lưu file output
SASREC_CAND_PATH = os.path.join(WORKING_DIR, 'sasrec_candidates.parquet')
LIGHTGCN_CAND_PATH = os.path.join(WORKING_DIR, 'lightgcn_candidates.parquet')
CAND_PATH = os.path.join(WORKING_DIR, 'candidates_phase2.parquet')

MAX_LEN   = 50


def load_data(path):
    df = pl.read_parquet(
        path,
        columns=['mapped_user_id', 'mapped_item_id', 'rating', 'timestamp']
    ).to_pandas()
    return df

In [ ]:
torch.cuda.empty_cache()
gc.collect()

In [ ]:
import torch.nn.functional as F
print("Đang tiền xử lý chuỗi bằng Polars...")
df_train_pl = pl.read_parquet(TRAIN_PATH, columns=['mapped_user_id', 'mapped_item_id', 'timestamp'])

In [ ]:
num_users = df_train_pl['mapped_user_id'].max() + 1
num_items = df_train_pl['mapped_item_id'].max() + 1

user_seqs_pl = (
    df_train_pl.sort(['mapped_user_id', 'timestamp'])
    .group_by('mapped_user_id')
    .agg(pl.col('mapped_item_id'))
)

mapped_user_ids = user_seqs_pl['mapped_user_id'].to_numpy()
item_lists = user_seqs_pl['mapped_item_id'].to_list()
X_sas_train = np.zeros((len(item_lists), MAX_LEN), dtype=np.int32)
for idx, seq in enumerate(item_lists):
    s = seq[-MAX_LEN:]
    X_sas_train[idx, MAX_LEN-len(s):] = s

del df_train_pl, user_seqs_pl, item_lists
gc.collect()
class SASRec(nn.Module):
    def __init__(self, n_items, embed_dim, max_len):
        super().__init__()
        self.item_emb = nn.Embedding(n_items, embed_dim, padding_idx=0)
        self.pos_emb = nn.Embedding(max_len, embed_dim)

        # SỬA Ở ĐÂY: Thêm norm_first=True để Transformer dễ hội tụ, không bị kẹt Loss
        layer = nn.TransformerEncoderLayer(
            d_model=embed_dim,
            nhead=1,
            batch_first=True,
            dim_feedforward=embed_dim*2,
            norm_first=True
        )
        self.transformer = nn.TransformerEncoder(layer, num_layers=1)

    def forward(self, seqs):
        pos = torch.arange(seqs.size(1), device=seqs.device).unsqueeze(0).expand_as(seqs)
        mask = (seqs == 0)
        out = self.transformer(self.item_emb(seqs) + self.pos_emb(pos), src_key_padding_mask=mask)
        return out[:, -1, :]

model_sasrec = SASRec(num_items, EMBED_DIM, MAX_LEN).to(device)

if torch.cuda.device_count() > 1:
    print(f"Đang sử dụng {torch.cuda.device_count()} GPUs cho SASRec!")
    model_sasrec = nn.DataParallel(model_sasrec)

optimizer = torch.optim.Adam(model_sasrec.parameters(), lr=0.001)
scaler = torch.cuda.amp.GradScaler()

model_sasrec.train()
epochs = 20
batch_size = 4096
EMBED_DIM = 64

print("Đang chuyển toàn bộ dữ liệu lên VRAM...")
X_tensor = torch.tensor(X_sas_train, dtype=torch.long, device=device)
# Tự động dọn rác X_sas_train nếu biến này còn tồn tại trên CPU
if 'X_sas_train' in locals():
    del X_sas_train; gc.collect()

for ep in range(epochs):
    idx_perm = torch.randperm(len(X_tensor), device=device)
    loss_ep, t_batches = 0, 0
    pbar = tqdm(range(0, len(X_tensor), batch_size), desc=f"Epoch {ep+1}/{epochs}", leave=False)

    for i in pbar:
        b_idx = idx_perm[i:i+batch_size]
        batch_seqs = X_tensor[b_idx]

        optimizer.zero_grad(set_to_none=True)
        
        with torch.amp.autocast('cuda'):
            u_reps = model_sasrec(batch_seqs[:, :-1])

            pos_items = batch_seqs[:, -1]
            neg_items = torch.randint(1, num_items, (len(batch_seqs),), device=device)

            # 2. Sửa lỗi AttributeError: Thêm .module để gọi đúng lớp Embedding
            pos_embs = model_sasrec.module.item_emb(pos_items)
            neg_embs = model_sasrec.module.item_emb(neg_items)

            pos_logits = (u_reps * pos_embs).sum(dim=-1)
            neg_logits = (u_reps * neg_embs).sum(dim=-1)

            labels_pos = torch.ones_like(pos_logits)
            labels_neg = torch.zeros_like(neg_logits)

            pos_loss = F.binary_cross_entropy_with_logits(pos_logits, labels_pos)
            neg_loss = F.binary_cross_entropy_with_logits(neg_logits, labels_neg)

            loss = pos_loss + neg_loss

        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()

        loss_ep += loss.item()
        t_batches += 1
        pbar.set_postfix(loss=loss_ep/t_batches)

    if (ep+1) % 2 == 0: print(f"SASRec Epoch {ep+1} Average Loss: {loss_ep/t_batches:.4f}")

In [ ]:
import os
import gc
import polars as pl
EMBED_DIM = 
torch.cuda.empty_cache()
gc.collect()

model_sasrec.eval()

infer_batch_size = 512
chunk_size = 1000 

print(f"Đang truy xuất Top 100 trực tiếp trên GPU (Batch size: {infer_batch_size})...")

# Tạo thư mục tạm để chứa các phần nhỏ
os.makedirs('/kaggle/working/sasrec_chunks', exist_ok=True)

all_top_idx = []
chunk_user_ids = []
chunk_idx = 0

with torch.no_grad():
    i_embs = torch.nn.functional.normalize(model_sasrec.module.item_emb.weight[1:], p=2, dim=1)

    pbar = tqdm(range(0, len(X_tensor), infer_batch_size), desc="Inference Native PyTorch")
    for i in pbar:
        with torch.amp.autocast('cuda'):
            u_reps = model_sasrec.module(X_tensor[i:i+infer_batch_size])
            u_reps = torch.nn.functional.normalize(u_reps, p=2, dim=1)
            
            scores = torch.matmul(u_reps, i_embs.T)
            _, top_idx = torch.topk(scores, 100, dim=1)

        all_top_idx.append(top_idx.cpu().numpy().astype('int32'))
        chunk_user_ids.append(mapped_user_ids[i:i+infer_batch_size])

        del scores, u_reps, top_idx

        # Nếu gom đủ 1000 batch hoặc đã chạy đến batch cuối cùng -> Tiến hành lưu ra đĩa
        if len(all_top_idx) >= chunk_size or (i + infer_batch_size) >= len(X_tensor):
            u_ids_arr = np.concatenate(chunk_user_ids)
            item_ids_arr = np.vstack(all_top_idx).flatten() + 1
            
            df_chunk = pd.DataFrame({
                'mapped_user_id': np.repeat(u_ids_arr, 100).astype('int32'),
                'mapped_item_id': item_ids_arr.astype('int32'),
                # Ép kiểu int8 cho rank (vì chỉ từ 1-100) để cực kỳ tiết kiệm RAM
                'sasrec_rank': np.tile(np.arange(1, 101, dtype=np.int8), len(u_ids_arr)) 
            })
            
            chunk_path = f'/kaggle/working/sasrec_chunks/chunk_{chunk_idx}.parquet'
            df_chunk.to_parquet(chunk_path)
            
            # Xóa sạch dữ liệu của chunk này khỏi RAM
            del df_chunk, u_ids_arr, item_ids_arr
            all_top_idx = []
            chunk_user_ids = []
            chunk_idx += 1
            gc.collect()

print("Đang dọn dẹp VRAM GPU...")
del X_tensor, i_embs
torch.cuda.empty_cache()
gc.collect()

print("Đang gộp các file nhỏ lại (không tốn RAM)...")
SASREC_CAND_PATH = '/kaggle/working/sasrec_candidates.parquet'

lf_sasrec = pl.scan_parquet('/kaggle/working/sasrec_chunks/chunk_*.parquet')
lf_sasrec.sink_parquet(SASREC_CAND_PATH)

print(f'Đã lưu kết quả SASRec hoàn chỉnh vào: {SASREC_CAND_PATH}')